In [ ]:
#rfe

In [ ]:
import pandas as pd
import lightgbm as lgb
import os
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTETomek, SMOTEENN
import warnings
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '6'

file_path = '../数据/训练集.csv'

if not os.path.exists(file_path):
    print(f"错误: 未找到文件 {file_path}")
else:
    df = pd.read_csv(file_path)
    target = 'ms_cds'
    X = df.drop(columns=[target])
    y = df[target]

    print("运行RFE ")
    estimator_for_rfe = xgb.XGBClassifier(n_jobs=-1, random_state=42,class_weight='balanced')
    rfe = RFE(estimator=estimator_for_rfe, n_features_to_select=1, step=0.05)
    rfe.fit(X, y)

    ranking_series = pd.Series(rfe.ranking_, index=X.columns)
    ranked_features_rfe = ranking_series.sort_values().index.tolist()
    print("RFE 特征排序完成。开始进入采样交叉验证循环...\n")

    samplers = {
        'SMOTE': SMOTE(random_state=42),
        'Tomek Links': TomekLinks(n_jobs=-1),
        'Smote+Tomek links': SMOTETomek(random_state=42, n_jobs=-1),
        'Smote Enn': SMOTEENN(random_state=42, n_jobs=-1)
    }


    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for name, sampler in samplers.items():
        print(f"正在分析采样方法: 【{name}】 ...")

        try:
            best_avg_f1 = -1
            best_k = 0
            best_features = []
            best_avg_metrics = {}

            for k in range(15, 26):
                if k > len(ranked_features_rfe):
                    break

                selected_feats = ranked_features_rfe[:k]

                acc_list, prec_list, rec_list, f1_list, auc_list = [], [], [], [], []

                for train_index, val_index in skf.split(X, y):
                    X_train_fold = X.iloc[train_index][selected_feats]
                    y_train_fold = y.iloc[train_index]
                    X_val_fold = X.iloc[val_index][selected_feats]
                    y_val_fold = y.iloc[val_index]
                    if sampler is not None:
                        X_res, y_res = sampler.fit_resample(X_train_fold, y_train_fold)
                    else:
                        X_res, y_res = X_train_fold, y_train_fold
                    clf = xgb.XGBClassifier(n_jobs=-1, random_state=42)
                    clf.fit(X_res, y_res)

                    y_pred = clf.predict(X_val_fold)
                    if hasattr(clf, "predict_proba"):
                        y_prob = clf.predict_proba(X_val_fold)[:, 1]
                    else:
                        y_prob = y_pred

                    # 计算指标
                    acc_list.append(accuracy_score(y_val_fold, y_pred))
                    prec_list.append(precision_score(y_val_fold, y_pred))
                    rec_list.append(recall_score(y_val_fold, y_pred))
                    f1_list.append(f1_score(y_val_fold, y_pred))
                    try:
                        auc_list.append(roc_auc_score(y_val_fold, y_prob))
                    except:
                        auc_list.append(float('nan'))

                # 计算平均值
                avg_acc = sum(acc_list) / len(acc_list)
                avg_prec = sum(prec_list) / len(prec_list)
                avg_rec = sum(rec_list) / len(rec_list)
                avg_f1 = sum(f1_list) / len(f1_list)
                # 处理AUC可能的缺失
                valid_auc = [x for x in auc_list if not pd.isna(x)]
                avg_auc = sum(valid_auc) / len(valid_auc) if valid_auc else float('nan')

                # 更新最佳k (基于平均F1)
                if avg_f1 > best_avg_f1:
                    best_avg_f1 = avg_f1
                    best_k = k
                    best_features = selected_feats
                    best_avg_metrics = {
                        'accuracy': avg_acc,
                        'precision': avg_prec,
                        'recall': avg_rec,
                        'f1': avg_f1,
                        'auc': avg_auc
                    }

            # 输出最佳结果
            if best_avg_f1 != -1:
                print(f"\n>>> 采样方法: {name} 的最佳表现 (特征数: {best_k}) <<<")
                print(f"平均准确率 (Accuracy):  {best_avg_metrics['accuracy']:.6f}")
                print(f"平均精确率 (Precision): {best_avg_metrics['precision']:.6f}")
                print(f"平均召回率 (Recall):    {best_avg_metrics['recall']:.6f}")
                print(f"平均F1-Score:           {best_avg_metrics['f1']:.6f}")
                if not pd.isna(best_avg_metrics['auc']):
                    print(f"平均AUC值:              {best_avg_metrics['auc']:.6f}")
                else:
                    print(f"平均AUC值:              无法计算")
                print(f"特征列表 ({best_k}个):")
                print(best_features)
                print("="*60 + "\n")
            else:
                print(f">>> {name} 未能生成有效模型结果。\n")

        except Exception as e:
            print(f">>> {name} 运行出错: {e}\n")
            print("="*60 + "\n")

In [ ]:
# 方差选择法

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTETomek, SMOTEENN

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '6'

file_path = '../数据/训练集.csv'

if not os.path.exists(file_path):
    print(f"错误: 未找到文件 {file_path}")
else:
    df = pd.read_csv(file_path)
    target = 'ms_cds'
    X = df.drop(columns=[target])
    y = df[target]

    print("正在执行方差选择法")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)          # 全数据归一化
    X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
    variances = X_scaled_df.var()
    ranked_features_global = variances.sort_values(ascending=False).index.tolist()
    print("方差特征排序完成。\n")

    samplers = {
        'SMOTE': SMOTE(random_state=42),
        'Tomek Links': TomekLinks(n_jobs=-1),
        'Smote+Tomek links': SMOTETomek(random_state=42, n_jobs=-1),
        'Smote Enn': SMOTEENN(random_state=42, n_jobs=-1)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, sampler in samplers.items():
        print(f"正在分析采样方法: 【{name}】 ...")

        best_avg_f1 = -1
        best_k = 0
        best_features = []
        best_scores = {}

        for k in range(15, 26):
            if k > len(ranked_features_global):
                break

            selected_feats = ranked_features_global[:k]

            acc_list = []
            prec_list = []
            rec_list = []
            f1_list = []
            auc_list = []

            # 交叉验证
            for train_idx, val_idx in cv.split(X, y):
                X_train_fold = X.iloc[train_idx][selected_feats]
                y_train_fold = y.iloc[train_idx]
                X_val_fold = X.iloc[val_idx][selected_feats]
                y_val_fold = y.iloc[val_idx]

                # 采样（仅在训练集上进行）
                if sampler is not None:
                    X_res, y_res = sampler.fit_resample(X_train_fold, y_train_fold)
                else:
                    X_res, y_res = X_train_fold, y_train_fold

                # 训练模型
                clf = xgb.XGBClassifier(n_jobs=-1, random_state=42)
                clf.fit(X_res, y_res)

                # 预测
                y_pred = clf.predict(X_val_fold)
                if hasattr(clf, "predict_proba"):
                    y_prob = clf.predict_proba(X_val_fold)[:, 1]
                else:
                    y_prob = y_pred

                # 计算指标
                acc_list.append(accuracy_score(y_val_fold, y_pred))
                prec_list.append(precision_score(y_val_fold, y_pred, zero_division=0))
                rec_list.append(recall_score(y_val_fold, y_pred, zero_division=0))
                f1_list.append(f1_score(y_val_fold, y_pred, zero_division=0))
                try:
                    auc_list.append(roc_auc_score(y_val_fold, y_prob))
                except:
                    auc_list.append(np.nan)

            avg_acc = np.nanmean(acc_list) if acc_list else 0
            avg_prec = np.nanmean(prec_list) if prec_list else 0
            avg_rec = np.nanmean(rec_list) if rec_list else 0
            avg_f1 = np.nanmean(f1_list) if f1_list else 0
            avg_auc = np.nanmean(auc_list) if auc_list else 0

            # 更新最佳结果
            if avg_f1 > best_avg_f1:
                best_avg_f1 = avg_f1
                best_k = k
                best_features = selected_feats
                best_scores = {
                    'accuracy': avg_acc,
                    'precision': avg_prec,
                    'recall': avg_rec,
                    'f1': avg_f1,
                    'auc': avg_auc
                }

        if best_features:
            print(f"\n>>> 采样方法: {name} | 10折交叉验证最佳表现 (特征数: {best_k}) <<<")
            print(f"准确率 (Accuracy):  {best_scores['accuracy']:.6f}")
            print(f"精确率 (Precision): {best_scores['precision']:.6f}")
            print(f"召回率 (Recall):    {best_scores['recall']:.6f}")
            print(f"F1-Score:           {best_scores['f1']:.6f}")
            print(f"AUC值:              {best_scores['auc']:.6f}")
            print(f"特征列表 ({best_k}个):")
            print(best_features)
            print("="*60 + "\n")
        else:
            print(f">>> {name} 未能生成有效模型结果。\n")
            print("="*60 + "\n")

In [ ]:
# mic

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '6'

file_path = '../数据/训练集.csv'

if not os.path.exists(file_path):
    print(f"错误: 未找到文件 {file_path}")
else:
    df = pd.read_csv(file_path)
    target = 'ms_cds'
    X = df.drop(columns=[target])
    y = df[target]

    print("正在计算互信息.")
    mi = mutual_info_classif(X, y, random_state=42)
    mi_series = pd.Series(mi, index=X.columns)
    ranked_features_global = mi_series.sort_values(ascending=False).index.tolist()
    ranked_features_global = [feat for feat in ranked_features_global if feat not in ['gender','hb4per']]
    print("特征排序完成，开始进入采样交叉验证循环...\n")

    samplers = {
        'SMOTE': SMOTE(random_state=42),
        'Tomek Links': TomekLinks(n_jobs=-1),
        'Smote+Tomek links': SMOTETomek(random_state=42, n_jobs=-1),
        'Smote Enn': SMOTEENN(random_state=42, n_jobs=-1)
    }

    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for name, sampler in samplers.items():
        print(f"正在分析采样方法: 【{name}】 ...")

        try:
            best_avg_f1 = -1
            best_k = 0
            best_features = []
            best_avg_metrics = {}

            for k in range(15, 26):
                if k > len(ranked_features_global):
                    break

                selected_feats = ranked_features_global[:k]

                acc_list, prec_list, rec_list, f1_list, auc_list = [], [], [], [], []

                for train_index, val_index in skf.split(X, y):
                    X_train_fold = X.iloc[train_index][selected_feats]
                    y_train_fold = y.iloc[train_index]
                    X_val_fold = X.iloc[val_index][selected_feats]
                    y_val_fold = y.iloc[val_index]

                    if sampler is not None:
                        X_res, y_res = sampler.fit_resample(X_train_fold, y_train_fold)
                    else:
                        X_res, y_res = X_train_fold, y_train_fold

                    clf = xgb.XGBClassifier(n_jobs=-1, random_state=42)
                    clf.fit(X_res, y_res)

                    y_pred = clf.predict(X_val_fold)
                    if hasattr(clf, "predict_proba"):
                        y_prob = clf.predict_proba(X_val_fold)[:, 1]
                    else:
                        y_prob = y_pred

                    # 计算指标
                    acc_list.append(accuracy_score(y_val_fold, y_pred))
                    prec_list.append(precision_score(y_val_fold, y_pred))
                    rec_list.append(recall_score(y_val_fold, y_pred))
                    f1_list.append(f1_score(y_val_fold, y_pred))
                    try:
                        auc_list.append(roc_auc_score(y_val_fold, y_prob))
                    except:
                        auc_list.append(float('nan'))

                avg_acc = np.mean(acc_list)
                avg_prec = np.mean(prec_list)
                avg_rec = np.mean(rec_list)
                avg_f1 = np.mean(f1_list)
                valid_auc = [x for x in auc_list if not pd.isna(x)]
                avg_auc = np.mean(valid_auc) if valid_auc else float('nan')

                if avg_f1 > best_avg_f1:
                    best_avg_f1 = avg_f1
                    best_k = k
                    best_features = selected_feats
                    best_avg_metrics = {
                        'accuracy': avg_acc,
                        'precision': avg_prec,
                        'recall': avg_rec,
                        'f1': avg_f1,
                        'auc': avg_auc
                    }

            if best_avg_f1 != -1:
                print(f"\n>>> 采样方法: {name} 的最佳表现 (特征数: {best_k}) <<<")
                print(f"平均准确率 (Accuracy):  {best_avg_metrics['accuracy']:.6f}")
                print(f"平均精确率 (Precision): {best_avg_metrics['precision']:.6f}")
                print(f"平均召回率 (Recall):    {best_avg_metrics['recall']:.6f}")
                print(f"平均F1-Score:           {best_avg_metrics['f1']:.6f}")
                if not pd.isna(best_avg_metrics['auc']):
                    print(f"平均AUC值:              {best_avg_metrics['auc']:.6f}")
                else:
                    print(f"平均AUC值:              无法计算")
                print(f"特征列表 ({best_k}个):")
                print(best_features)
                print("="*60 + "\n")
            else:
                print(f">>> {name} 未能生成有效模型结果。\n")

        except Exception as e:
            print(f">>> {name} 运行出错: {e}\n")
            print("="*60 + "\n")